# Project 2 figures
Run from top to bottom. Change the first cell for shared typography; each figure cell contains its layout and legends. PDFs are written here and copied to the thesis. Structure artwork requires GIMP 3.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys, shutil, logging, json, csv, re, io, os, subprocess, tempfile, hashlib
sys.dont_write_bytecode = True
import xml.etree.ElementTree as ET
from functools import lru_cache
import numpy as np
import h5py, yaml, fitz
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display
from PIL import Image

OUT = Path.cwd()
if not (OUT / 'style.py').exists():
    OUT = OUT / 'figures_for_thesis'
assert (OUT / 'style.py').exists(), 'Open this notebook from its folder or the repository root.'
ROOT = OUT.parent
THESIS = ROOT.parent / 'PhD_thesis_20251216' / 'figures_proj2'
sys.path[:0] = [str(OUT), str(ROOT)]
from electronic_data import bands, dos
from vmatplot.phonon import extract_phonon_bands, extract_qpath
from vmatplot.algorithms import fit_birch_murnaghan

# Shared typography and colours.
BLUE, GREEN, YELLOW, ORANGE = '#1478E1', '#28AF3C', '#FAC828', '#FA8C00'
PURPLE, CYAN, GREY = '#8C64E1', '#32B4C8', '#787878'
plt.rcParams.update({'text.usetex': False, 'font.family': 'serif', 'mathtext.fontset': 'cm',
 'axes.labelsize': 16, 'xtick.labelsize': 14, 'ytick.labelsize': 14,
 'legend.fontsize': 14, 'figure.dpi': 196, 'figure.facecolor': 'w',
 'lines.linewidth': 1.5, 'lines.solid_capstyle': 'round', 'lines.dash_capstyle': 'round',
 'lines.solid_joinstyle': 'round', 'lines.dash_joinstyle': 'round', 'pdf.fonttype': 42})
logging.getLogger('fontTools').setLevel(logging.ERROR)


Shared panel helpers

In [ ]:
def tab(ax, text):
    ax.set_title(text, loc='left', x=.035, y=.96, pad=0, va='top', fontsize=12,
                 bbox={'boxstyle': 'round', 'facecolor': 'white',
                       'edgecolor': plt.rcParams['legend.edgecolor'],
                       'alpha': plt.rcParams['legend.framealpha']}, zorder=10)


def frame(ax):
    ax.tick_params(direction='in', which='both', top=True, right=True)


def legend(fig, ax, ncol=2):
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower left', bbox_to_anchor=(.08, .012),
               ncol=ncol, frameon=True, fancybox=True)



def save(fig, name):
    fig.savefig(OUT / name, metadata={'CreationDate': None})
    shutil.copy2(OUT / name, THESIS / name)
    display(fig)
    plt.close(fig)

def draw_bands(ax, folder, spin=False, color=BLUE, label=None, linestyle='-'):
    x, energies, ticks, labels, breaks, _ = bands(folder)
    for channel, values in enumerate(energies if spin else energies[:1]):
        xx, yy = x.copy(), values.copy()
        for i in reversed(breaks):
            xx = np.insert(xx, i, xx[i]); yy = np.insert(yy, i, np.nan, axis=0)
        line = ax.plot(xx, yy, color=(PURPLE, CYAN)[channel] if spin else color,
                       ls=('-', (0,(4,3)))[channel] if spin else linestyle)
        if label: line[0].set_label(label)
    for tick in ticks[1:-1]: ax.axvline(tick, color=GREY, ls='--', alpha=.4, zorder=0)
    ax.axhline(0, color=GREY, ls='--', zorder=1)
    ax.set_xticks(ticks, labels); ax.set_xlim(x[0], x[-1]); ax.set_ylim(-4, 3)
    ax.set_xlabel(r'Wave vector ($k$)'); frame(ax)

def spin_legend(fig, ax, bilayer=False):
    handles = [Line2D([],[],color=PURPLE,label='Spin up'),
               Line2D([],[],color=CYAN,ls=(0,(4,3)),label='Spin down')]
    if bilayer: handles.append(Line2D([],[],color=BLUE,label='Bilayer bands'))
    handles.append(Line2D([],[],color=GREY,ls='--',label=r'$E_{\mathrm{F}}=0$'))
    fig.legend(handles=handles,loc='lower left',bbox_to_anchor=(.085,.015),
               ncol=len(handles),frameon=True,fancybox=True)


fig2.6 — HSE06 bands

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(10,4.6),sharey=True)

for ax, folder, title in zip(axes, ['monolayer_FM_HSE06','monolayer_AFM_HSE06','bilayer_HSE'],
                            ['(a) FM monolayer','(b) AFM monolayer','(c) Bilayer']):
    draw_bands(ax,folder,spin=folder!='bilayer_HSE'); tab(ax,title)

axes[0].set_ylabel(r'$E-E_{\mathrm{F}}$ (eV)')

spin_legend(fig,axes[0],True)

fig.subplots_adjust(left=.085,right=.985,bottom=.25,top=.95,wspace=.09)

save(fig,'fig2.6.pdf')


S2.11 — FM PBE/HSE06 bands

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(10,4.7),sharey=True)

for ax,folder,title in zip(axes,['monolayer_FM','monolayer_FM_HSE06'],['(a) GGA-PBE','(b) HSE06']):
    draw_bands(ax,folder,spin=True); tab(ax,title)

axes[0].set_ylabel(r'$E-E_{\mathrm{F}}$ (eV)')

spin_legend(fig,axes[0])

fig.subplots_adjust(left=.085,right=.985,bottom=.24,top=.95,wspace=.08)

save(fig,'S2.11.pdf')


S2.13 and S2.14 — functional and symmetry comparisons

In [ ]:
for filename, folders, labels in [
 ('S2.13.pdf',['monolayer','monolayer_HSE','monolayer_R2SCAN'],['GGA-PBE','HSE06','R2SCAN']),
 ('S2.14.pdf',['monolayer','monolayer_shifting','monolayer_sym_off'],['Original structure','Shifted atoms','Symmetry off'])]:
    fig, ax = plt.subplots(figsize=(10,4.8))
    for folder,label,color in zip(folders,labels,[BLUE,ORANGE,'#8CAF28']):
        draw_bands(ax,folder,color=color,label=label)
    ax.set_ylim(-4,4); ax.set_ylabel(r'$E-E_{\mathrm{F}}$ (eV)')
    tab(ax,'Monolayer'); legend(fig,ax,3)
    fig.subplots_adjust(left=.09,right=.985,bottom=.24,top=.95)
    save(fig,filename)


S2.16 — FM/AFM spin DOS

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(10,4.8),sharey=True)

for ax, folder, title in zip(axes,['monolayer_FM_ollie','monolayer_AFM_ollie'],['(a) FM','(b) AFM']):
    energy, channels, _ = dos(folder)
    ax.plot(energy,channels[0]+channels[1],color=BLUE,label='Total')
    ax.plot(energy,channels[0],color=ORANGE,label='Spin up')
    ax.plot(energy,-channels[1],color=CYAN,label='Spin down')
    ax.axvline(0,color=GREY,ls='--',label=r'$E_{\mathrm{F}}=0$')
    ax.set_xlim(-14,6); ax.set_ylim(-8,14); ax.set_xlabel(r'$E-E_{\mathrm{F}}$ (eV)')
    tab(ax,title); frame(ax)

axes[0].set_ylabel('Density of states')

legend(fig,axes[0],4)

fig.subplots_adjust(left=.085,right=.985,bottom=.24,top=.95,wspace=.08)

save(fig,'S2.16.pdf')


S2.12 — total DOS

In [ ]:
fig, ax = plt.subplots(figsize=(10,4.8))

for folder,label,color in [('o-B14_K20','Bulk',BLUE),('monolayer','Monolayer',GREEN),
                            ('bilayer','Bilayer',YELLOW),('bilayer_with_Hydrogen','H-terminated bilayer',ORANGE)]:
    energy, channels, _ = dos(folder)
    ax.plot(energy,channels[0],color=color,label=label)

ax.axvline(0,color=GREY,ls='--')

ax.set(xlim=(-6,6),ylim=(0,26),
           xlabel=r'$E-E_{\mathrm{F}}$ (eV)',ylabel='Density of states')

frame(ax)

legend(fig,ax,2)

fig.subplots_adjust(left=.085,right=.985,bottom=.28,top=.95)

save(fig,'S2.12.pdf')


fig2.8 — H-bilayer bands and DOS

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(10,5),gridspec_kw={'width_ratios':[3,1]},sharey=True)

draw_bands(axes[0],'bilayer_with_Hydrogen')

axes[0].set_ylim(-4,4)

energy, channels, _ = dos('bilayer_with_Hydrogen')

axes[1].plot(channels[0],energy,color=BLUE)

axes[1].axhline(0,color=GREY,ls='--')

axes[1].set_xlim(0,20)

axes[0].set_ylabel(r'$E-E_{\mathrm{F}}$ (eV)')

axes[1].set_xlabel('DOS (a.u.)')

tab(axes[0],'(a) Bands')

tab(axes[1],'(b) DOS')

frame(axes[1])

fig.subplots_adjust(left=.085,right=.985,bottom=.17,top=.95,wspace=.10)

save(fig,'fig2.8.pdf')


Optical data and derived quantities

In [ ]:
systems = [('bilayer_with_Hydrogen', 'H-terminated bilayer', ORANGE, 30.088116646 / 11.208620),
           ('bilayer', 'Bilayer', YELLOW, 28.088116646 / 9.41157),
           ('monolayer', 'Monolayer', GREEN, 24.044058323 / 5.807056948202938),
           ('o-B14_n128_k34', 'Bulk', BLUE, 1.)]


def dielectric(folder, factor=1.):
    with h5py.File(ROOT / '5.1_dielectric_function' / folder / 'vaspout.h5') as f:
        group = f['results/linear_response']
        energy = group['energies_dielectric_function'][:]
        tensor = group['density_density_dielectric_function'][:]
    real = np.array([tensor[i, i, :, 0] for i in range(3)]).T
    imag = np.array([tensor[i, i, :, 1] for i in range(3)]).T
    return energy, 1 + factor * (real - 1), factor * imag


def quantity(energy, real, imag, kind):
    modulus = np.hypot(real, imag)
    n = np.sqrt(np.maximum(0, (modulus + real) / 2))
    k = np.sqrt(np.maximum(0, (modulus - real) / 2))
    return {'real': real, 'imag': imag, 'n': n, 'k': k,
            'alpha': 2 * energy[:, None] * k / (6.582119569e-16 * 2.99792458e18),
            'loss': imag / (real**2 + imag**2),
            'R': ((n - 1)**2 + k**2) / ((n + 1)**2 + k**2)}[kind]



optical_data = {folder: dielectric(folder, factor) for folder, _, _, factor in systems}


fig2.10 — optical response

In [ ]:
name = 'fig2.10.pdf'
rows = [('real', r'$\varepsilon_1$'), ('imag', r'$\varepsilon_2$')]
labels = ['a', 'b']
fig, axes = plt.subplots(len(rows), 3, figsize=(10, 3.0 * len(rows) + 1.0), squeeze=False)
for row, (kind, ylabel) in enumerate(rows):
    for col, component in enumerate(('xx', 'yy', 'zz')):
        ax = axes[row, col]
        for folder, label, color, factor in systems:
            energy, real, imag = optical_data[folder]
            selected = (energy >= 0) & (energy <= 12)
            ax.plot(energy[selected], quantity(energy, real, imag, kind)[selected, col],
                    color=color, label=label)
        if kind == 'real':
            lo, hi = ax.get_ylim(); ax.set_ylim(max(-60, lo), min(60, hi))
        elif kind == 'imag':
            hi = min(60, ax.get_ylim()[1]); ax.set_ylim(-.05*hi, hi)
        ax.set_xlim(0, 12)
        ax.set_xticks([0, 4, 8, 12])
        ax.ticklabel_format(axis='y', style='plain', useOffset=False)
        frame(ax)
        tab(ax, f'({labels[row]}) {component}' if col == 0 else component)
        if col == 0: ax.set_ylabel(ylabel)
        if row == len(rows) - 1: ax.set_xlabel('Photon energy (eV)')
        else: ax.tick_params(labelbottom=False)
legend(fig, axes[0, 0])
fig.subplots_adjust(left=.11, right=.985, bottom=.32 if len(rows) == 1 else .23,
                    top=.95, wspace=.27, hspace=.13)
save(fig, name)


fig2.11 — optical response

In [ ]:
name = 'fig2.11.pdf'
rows = [('alpha', r'Absorption ($\mathrm{\AA}^{-1}$)'), ('loss', 'Energy loss')]
labels = ['a', 'b']
fig, axes = plt.subplots(len(rows), 3, figsize=(10, 3.0 * len(rows) + 1.0), squeeze=False)
for row, (kind, ylabel) in enumerate(rows):
    for col, component in enumerate(('xx', 'yy', 'zz')):
        ax = axes[row, col]
        for folder, label, color, factor in systems:
            energy, real, imag = optical_data[folder]
            selected = (energy >= 0) & (energy <= 12)
            ax.plot(energy[selected], quantity(energy, real, imag, kind)[selected, col],
                    color=color, label=label)
        if kind == 'real':
            lo, hi = ax.get_ylim(); ax.set_ylim(max(-60, lo), min(60, hi))
        elif kind == 'imag':
            hi = min(60, ax.get_ylim()[1]); ax.set_ylim(-.05*hi, hi)
        ax.set_xlim(0, 12)
        ax.set_xticks([0, 4, 8, 12])
        ax.ticklabel_format(axis='y', style='plain', useOffset=False)
        frame(ax)
        tab(ax, f'({labels[row]}) {component}' if col == 0 else component)
        if col == 0: ax.set_ylabel(ylabel)
        if row == len(rows) - 1: ax.set_xlabel('Photon energy (eV)')
        else: ax.tick_params(labelbottom=False)
legend(fig, axes[0, 0])
fig.subplots_adjust(left=.11, right=.985, bottom=.32 if len(rows) == 1 else .23,
                    top=.95, wspace=.27, hspace=.13)
save(fig, name)


fig2.12 — optical response

In [ ]:
name = 'fig2.12.pdf'
rows = [('R', 'Reflectivity'), ('n', 'Refractive index')]
labels = ['a', 'b']
fig, axes = plt.subplots(len(rows), 3, figsize=(10, 3.0 * len(rows) + 1.0), squeeze=False)
for row, (kind, ylabel) in enumerate(rows):
    for col, component in enumerate(('xx', 'yy', 'zz')):
        ax = axes[row, col]
        for folder, label, color, factor in systems:
            energy, real, imag = optical_data[folder]
            selected = (energy >= 0) & (energy <= 12)
            ax.plot(energy[selected], quantity(energy, real, imag, kind)[selected, col],
                    color=color, label=label)
        if kind == 'real':
            lo, hi = ax.get_ylim(); ax.set_ylim(max(-60, lo), min(60, hi))
        elif kind == 'imag':
            hi = min(60, ax.get_ylim()[1]); ax.set_ylim(-.05*hi, hi)
        ax.set_xlim(0, 12)
        ax.set_xticks([0, 4, 8, 12])
        ax.ticklabel_format(axis='y', style='plain', useOffset=False)
        frame(ax)
        tab(ax, f'({labels[row]}) {component}' if col == 0 else component)
        if col == 0: ax.set_ylabel(ylabel)
        if row == len(rows) - 1: ax.set_xlabel('Photon energy (eV)')
        else: ax.tick_params(labelbottom=False)
legend(fig, axes[0, 0])
fig.subplots_adjust(left=.11, right=.985, bottom=.32 if len(rows) == 1 else .23,
                    top=.95, wspace=.27, hspace=.13)
save(fig, name)


S2.18 — optical response

In [ ]:
name = 'S2.18.pdf'
rows = [('k', 'Extinction coefficient')]
labels = ['a']
fig, axes = plt.subplots(len(rows), 3, figsize=(10, 3.0 * len(rows) + 1.0), squeeze=False)
for row, (kind, ylabel) in enumerate(rows):
    for col, component in enumerate(('xx', 'yy', 'zz')):
        ax = axes[row, col]
        for folder, label, color, factor in systems:
            energy, real, imag = optical_data[folder]
            selected = (energy >= 0) & (energy <= 12)
            ax.plot(energy[selected], quantity(energy, real, imag, kind)[selected, col],
                    color=color, label=label)
        if kind == 'real':
            lo, hi = ax.get_ylim(); ax.set_ylim(max(-60, lo), min(60, hi))
        elif kind == 'imag':
            hi = min(60, ax.get_ylim()[1]); ax.set_ylim(-.05*hi, hi)
        ax.set_xlim(0, 12)
        ax.set_xticks([0, 4, 8, 12])
        ax.ticklabel_format(axis='y', style='plain', useOffset=False)
        frame(ax)
        tab(ax, f'({labels[row]}) {component}' if col == 0 else component)
        if col == 0: ax.set_ylabel(ylabel)
        if row == len(rows) - 1: ax.set_xlabel('Photon energy (eV)')
        else: ax.tick_params(labelbottom=False)
legend(fig, axes[0, 0])
fig.subplots_adjust(left=.11, right=.985, bottom=.32 if len(rows) == 1 else .23,
                    top=.95, wspace=.27, hspace=.13)
save(fig, name)


S2.4 — optical k-mesh convergence

In [ ]:
name = 'S2.4.pdf'
folders = [f'o-B14_n128_k{k}' for k in (10,20,26,30,32,34)]
labels = ['10×14×9', '20×28×18', '26×37×24', '30×42×28', '32×45×29', '34×48×31']
limits = [(1,8),(5,15),(0,4)]
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for folder, label, color in zip(folders, labels, [GREEN, '#E65050', ORANGE, YELLOW, CYAN, BLUE]):
    energy, real, imag = dielectric(folder)
    for row, values in enumerate((real, imag)):
        for col in range(3):
            axes[row,col].plot(energy, values[:,col], color=color, label=label)
for i, ax in enumerate(axes.flat):
    row, col = divmod(i,3)
    ax.set_xlim(limits[col]); frame(ax)
    tab(ax, ('xx','yy','zz')[col])
    if col == 0: ax.set_ylabel(r'$\varepsilon_1$' if row == 0 else r'$\varepsilon_2$')
    if row == 1: ax.set_xlabel('Photon energy (eV)')
    else: ax.tick_params(labelbottom=False)
    # Autoscale to the displayed energy interval, including all plotted datasets.
    shown = [line.get_ydata()[(line.get_xdata() >= limits[col][0]) & (line.get_xdata() <= limits[col][1])]
             for line in ax.lines]
    lo, hi = min(a.min() for a in shown), max(a.max() for a in shown)
    pad = .08 * (hi - lo)
    ax.set_ylim(lo-pad, hi+pad)
legend(fig, axes[0,0], ncol=3)
fig.subplots_adjust(left=.1,right=.985,bottom=.23,top=.95,wspace=.27,hspace=.13)
save(fig,name)


S2.5 — optical band-count convergence

In [ ]:
name = 'S2.5.pdf'
folders = [f'o-B14_n{k}_k10' for k in (32,64,128,256)]
labels = ['48 bands', '72 bands', '144 bands', '264 bands']
limits = [(15,25),(15,30),(10,30)]
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for folder, label, color in zip(folders, labels, [GREEN, '#E65050', ORANGE, YELLOW, CYAN, BLUE]):
    energy, real, imag = dielectric(folder)
    for row, values in enumerate((real, imag)):
        for col in range(3):
            axes[row,col].plot(energy, values[:,col], color=color, label=label)
for i, ax in enumerate(axes.flat):
    row, col = divmod(i,3)
    ax.set_xlim(limits[col]); frame(ax)
    tab(ax, ('xx','yy','zz')[col])
    if col == 0: ax.set_ylabel(r'$\varepsilon_1$' if row == 0 else r'$\varepsilon_2$')
    if row == 1: ax.set_xlabel('Photon energy (eV)')
    else: ax.tick_params(labelbottom=False)
    # Autoscale to the displayed energy interval, including all plotted datasets.
    shown = [line.get_ydata()[(line.get_xdata() >= limits[col][0]) & (line.get_xdata() <= limits[col][1])]
             for line in ax.lines]
    lo, hi = min(a.min() for a in shown), max(a.max() for a in shown)
    pad = .08 * (hi - lo)
    ax.set_ylim(lo-pad, hi+pad)
legend(fig, axes[0,0], ncol=3)
fig.subplots_adjust(left=.1,right=.985,bottom=.23,top=.95,wspace=.27,hspace=.13)
save(fig,name)


Phonon data

In [ ]:
@lru_cache(None)
def phonon(folder):
    directory=ROOT/folder
    if (directory/'band.yaml').exists():
        with open(directory/'band.yaml') as f: raw=yaml.load(f,Loader=yaml.CSafeLoader)
        x=np.array([q['distance'] for q in raw['phonon']])
        y=np.array([[b['frequency'] for b in q['band']] for q in raw['phonon']])
        assert np.all(np.diff(x)>=-1e-8)
        ends=np.cumsum(raw['segment_nqpoint'])
        ticks=[x[0]]
        for end in ends:
            if x[end-1]>ticks[-1]+1e-10: ticks.append(x[end-1])
        labels=['Γ','Z','T','Y','Γ']; assert len(ticks)==len(labels)
        for end in reversed(ends[:-1]):
            x=np.insert(x,end,np.nan); y=np.insert(y,end,np.nan,axis=0)
        return x,y,ticks,labels
    else:
        raw=extract_phonon_bands(str(directory)); x=np.array(extract_qpath(str(directory)))
        n=len(x)//4
        ticks=x[[0,n-1,2*n-1,3*n-1,len(x)-1]]; labels=['Γ','Z','T','Y','Γ']
    return x, np.array(raw['bands']).T, ticks, labels


def draw_phonon(ax,folder,label,color):
    x,y,ticks,labels=phonon(folder)
    lines=ax.plot(x,y,color=color); lines[0].set_label(label)
    ax.set_xticks(ticks,labels); ax.set_xlim(np.nanmin(x),np.nanmax(x))
    ax.set_xlabel(r'Wave vector ($q$)'); frame(ax)


pristine=[('3.0_phonon_dispersion_vasp/monolayer_3','Monolayer',CYAN),
          ('3.0_phonon_dispersion_vasp/bilayer_3','Bilayer',YELLOW)]
hydrogen=[('3.0_phonon_dispersion_phononpy/monolayer_top','H-terminated monolayer','#8CAF28'),
          ('3.0_phonon_dispersion_phononpy/bilayer_top','H-terminated bilayer',ORANGE)]


S2.9 — pristine and hydrogen-terminated phonons

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4.8),sharey=True)
for ax,group,title in zip(axes,[pristine,hydrogen],['(a) Pristine','(b) Hydrogen-terminated']):
    for folder,label,color in group: draw_phonon(ax,folder,label,color)
    ax.axhline(0,color=GREY,ls='--',zorder=0); ax.set_ylim(-2,8); tab(ax,title)
axes[0].set_ylabel('Frequency (THz)')
handles=sum([ax.get_legend_handles_labels()[0] for ax in axes],[])
labels=sum([ax.get_legend_handles_labels()[1] for ax in axes],[])
fig.legend(handles,labels,loc='lower left',bbox_to_anchor=(.08,.01),ncol=2,fancybox=True,frameon=True)
fig.subplots_adjust(left=.085,right=.985,bottom=.27,top=.95,wspace=.08)
save(fig,'S2.9.pdf')


fig2.5 and S2.8 — phonon comparisons

In [ ]:
for filename,group,ylim in [('fig2.5.pdf',pristine,(-1,7)),
 ('S2.8.pdf',[(f'3.0_phonon_dispersion_vasp/{folder}',label,color)
  for folder,label,color in [('monolayer_2','Monolayer',CYAN),('monolayer_H_2','H-terminated monolayer','#8CAF28'),
  ('bilayer_2','Bilayer',YELLOW),('bilayer_H_2','H-terminated bilayer',ORANGE)]],(-3,4))]:
    fig,ax=plt.subplots(figsize=(10,4.8))
    for folder,label,color in group: draw_phonon(ax,folder,label,color)
    ax.axhline(0,color=GREY,ls='--',zorder=0); ax.set_ylim(ylim); ax.set_ylabel('Frequency (THz)')
    legend(fig,ax,2); fig.subplots_adjust(left=.085,right=.985,bottom=.27,top=.95)
    save(fig,filename)


fig2.9 — superconducting gap distributions

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4.8),sharex=True,sharey=True)
cmap=LinearSegmentedColormap.from_list('gap_density',['white',BLUE])
records={}
for ax,folder,title in zip(axes,['bulk','bilayer_with_Hydrogen'],
                          ['(a) Bulk','(b) H-terminated bilayer']):
    rows=[]
    for file in sorted((ROOT/'superconductivity'/folder).glob('B14.imag_aniso_gap0_*')):
        temperature=float(file.name.rsplit('_',1)[1])
        data=np.loadtxt(file,ndmin=2)
        density=data[:,0]-temperature
        delta=data[:,1]
        assert np.isfinite(data).all() and np.all(density>=0)
        assert np.isclose(density.max(),1) and np.all(density<=1+1e-8)
        # Column 1 is T + rho/max(rho), not an independent temperature sample.
        points=ax.scatter(np.full(len(delta),temperature),delta,c=density,
                          cmap=cmap,vmin=0,vmax=1,marker='_',s=14,linewidths=.9)
        rows.append({'temperature_K':temperature,'bins':len(delta),
                     'gap_min_meV':float(delta.min()),'gap_max_meV':float(delta.max()),
                     'density_mean_meV':float(np.average(delta,weights=density)),
                     'density_min':float(density.min()),'density_max':float(density.max())})
    ax.plot([r['temperature_K'] for r in rows],[r['density_mean_meV'] for r in rows],
            color=ORANGE,label='Distribution mean')
    ax.set(xlim=(0,31),ylim=(0,6.6),xlabel='Temperature (K)')
    ax.set_xticks([0,10,20,30]); tab(ax,title); frame(ax)
    records[folder]=rows
axes[0].set_ylabel(r'Superconducting gap $\Delta$ (meV)')
legend(fig,axes[0],1)
fig.subplots_adjust(left=.09,right=.845,bottom=.24,top=.95,wspace=.08)
colorbar=fig.colorbar(points,cax=fig.add_axes([.875,.24,.025,.71]))
colorbar.set_label('Normalized gap density',fontsize=16)
colorbar.ax.tick_params(direction='in',labelsize=14)
save(fig,'fig2.9.pdf')
(OUT/'gap_summary.json').write_text(json.dumps(records,indent=2)+'\n')


Convergence data

In [ ]:
base=ROOT/'1.0_energy/o-B14'


def table(folder,name='energy_parameters.dat'):
    with open(base/folder/name) as f: rows=list(csv.DictReader(f,delimiter='\t'))
    aliases={'kpoints(x y z)':'kpoints mesh','encut':'energy cutoff (encut)'}
    return [{aliases.get(k.lower(),k.lower()):v for k,v in row.items()} for row in rows]


def series(folder,xkey='total kpoints',ykey='total energy',lower=40,name='energy_parameters.dat'):
    rows=table(folder,name)
    rows=sorted([r for r in rows if float(r[xkey])>=lower],key=lambda r:float(r[xkey]))
    return np.array([float(r[xkey]) for r in rows]),np.array([float(r[ykey]) for r in rows]),rows


S2.1 — total-energy convergence and lattice scan

In [ ]:
fig=plt.figure(figsize=(10,7.6)); gs=fig.add_gridspec(2,2,left=.155,right=.985,bottom=.13,top=.95,wspace=.28,hspace=.39)
axes=[fig.add_subplot(gs[0,0]),fig.add_subplot(gs[0,1]),fig.add_subplot(gs[1,:])]
for cutoff,color in [(450,GREEN),(480,BLUE)]:
    x,y,rows=series(f'energy_kpoints_{cutoff}')
    # Plot the actual first Monkhorst-Pack grid integer; the full meshes remain in the source table.
    nk=np.array([int(r['kpoints mesh'].strip('()').split(',')[0]) for r in rows])
    axes[0].plot(nk,y,'o-',ms=4,color=color,label=rf'$E_{{\rm cut}}={cutoff}$ eV')
for grid,label,color in [('1260',r'$10\times14\times9$',PURPLE),('8721',r'$19\times27\times17$','#D25ADC')]:
    x,y,_=series(f'energy_encut_{grid}','energy cutoff (encut)',lower=400)
    axes[1].plot(x,y,'o-',ms=3,color=color,label=label)
for i,ax in enumerate(axes[:2]):
    ax.set_ylabel('Energy (eV)'); ax.ticklabel_format(axis='y',style='plain',useOffset=False)
    frame(ax);tab(ax,['(a) k-point grid','(b) Energy cutoff'][i])
    ax.legend(loc='lower right' if i == 0 else 'upper right',fancybox=True,frameon=True)
axes[0].set_xlabel(r'Grid index ($n_x$)');axes[0].set_xticks([4,8,12,16,20,24])
axes[1].set_xlabel('Energy cutoff (eV)')
x,y,_=series('energy_lattice','lattice constant',lower=0)
params,xx,yy=fit_birch_murnaghan(x,y,sample_count=100)
axes[2].plot(xx,yy,color=BLUE,label='EOS fit')
axes[2].plot(x,y,'o',ms=4,color=BLUE,label='Sampled data')
imin=np.argmin(y);axes[2].axvline(x[imin],color=GREY,ls='--',label=rf'Minimum: {x[imin]:.5f} $\mathrm{{\AA}}$')
axes[2].set(xlabel=r'Lattice constant ($\mathrm{\AA}$)',ylabel='Energy (eV)')
axes[2].ticklabel_format(axis='both',style='plain',useOffset=False)
frame(axes[2]);tab(axes[2],'(c) Lattice constant');axes[2].legend(loc='upper right',frameon=True,fancybox=True)
save(fig,'S2.1.pdf')


S2.2 — cohesive-energy convergence

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4.8),sharey=True)
for ax,folder,xkey,color,lower,title in [
 (axes[0],'energy_kpoints_480','total kpoints',BLUE,40,'(a) k-point grid'),
 (axes[1],'energy_encut_1260','energy cutoff (encut)',PURPLE,400,'(b) Energy cutoff')]:
    rows=table(folder,'cohesive_energy.dat')
    ykey=next(k for k in rows[0] if 'cohesive' in k.lower())
    x,y,rows=series(folder,xkey,ykey,lower,'cohesive_energy.dat')
    if xkey=='total kpoints': x=np.array([int(r['kpoints mesh'].strip('()').split(',')[0]) for r in rows])
    ax.plot(x,y,'o-',ms=4,color=color)
    ax.ticklabel_format(axis='y',style='plain',useOffset=False);frame(ax);tab(ax,title)
axes[0].set_xticks([4,8,12,16,20,24]);axes[0].set_xlabel(r'Grid index ($n_x$)')
axes[1].set_xlabel('Energy cutoff (eV)');axes[0].set_ylabel('Cohesive energy (eV/atom)')
fig.subplots_adjust(left=.135,right=.985,bottom=.18,top=.95,wspace=.08)
save(fig,'S2.2.pdf')


S2.3 — supercell-height convergence

In [ ]:
with open(ROOT/'2.1_geometry_optimization/monolayer/energy_vacuum/energy_parameters.dat') as f:
    rows=list(csv.DictReader(f,delimiter='\t'))
x=np.array([float(r['a3']) for r in rows]); y=np.array([float(r['total energy']) for r in rows])
fig,ax=plt.subplots(figsize=(10,4.6));ax.plot(x,y,'o-',color=BLUE,ms=4)
ax.set(xlabel=r'Supercell height ($a_3$, $\mathrm{\AA}$)',ylabel='Energy (eV)')
ax.ticklabel_format(axis='y',style='plain',useOffset=False);frame(ax)
fig.subplots_adjust(left=.155,right=.985,bottom=.18,top=.95);save(fig,'S2.3.pdf')


S2.15 — fixed-spin energies

In [ ]:
# This supplied tabulation is the source of the thesis fixed-spin figure.
with open(OUT/'fixed_spin_data.csv') as f: rows=list(csv.DictReader(f))
fig,ax=plt.subplots(figsize=(10,4.6))
for method,color,label in [('PBE+D3',BLUE,'PBE+D3'),('r2SCAN',ORANGE,r'r$^2$SCAN')]:
    selected=[r for r in rows if r['method']==method]
    ax.plot([float(r['fixed_spin_muB']) for r in selected],
            [float(r['relative_energy_eV']) for r in selected],'o-',ms=4,color=color,label=label)
ax.set(xlabel=r'Fixed spin moment ($\mu_{\mathrm{B}}$)',ylabel='Relative energy (eV)')
frame(ax);legend(fig,ax,2);fig.subplots_adjust(left=.10,right=.985,bottom=.24,top=.95)
save(fig,'S2.15.pdf')


fig2.2 — bulk bands, Brillouin zone and PDOS

In [ ]:
# Bulk bands / Brillouin zone / grouped PDOS in one fixed-width figure.
fig=plt.figure(figsize=(10,8.2))
gs=fig.add_gridspec(2,2,width_ratios=[1.6,1],left=.095,right=.98,bottom=.23,top=.95,hspace=.28,wspace=.1)
ax=fig.add_subplot(gs[0,0]); bz=fig.add_subplot(gs[0,1]); pd=fig.add_subplot(gs[1,:]); bz.axis('off')
draw_bands(ax,'o-B14_K48');ax.set_ylim(-4,4);ax.set_ylabel(r'$E-E_{\mathrm{F}}$ (eV)')
tab(ax,'(a) Bands');tab(bz,'(b) Brillouin zone')
r=ET.parse(ROOT/'4.1_PDoS/o-B14_K20/vasprun.xml').getroot();section=r.find('.//dos')
fermi=float(section.find("i[@name='efermi']").text)
arrays=np.array([[[float(v) for v in row.text.split()] for row in ion.findall('./set/r')]
                 for ion in section.findall('./partial/array/set/set')])
energy=arrays[0,:,0]-fermi
for group,atoms,color in [(2,range(4,8),BLUE),(3,[0,1,2,3,12,13],ORANGE),(7,range(14),PURPLE)]:
    values=arrays[list(atoms)].sum(axis=0)
    pd.plot(energy,values[:,2:5].sum(axis=1),color=color,label=rf'G{group}: $p$')
    pd.plot(energy,values[:,1],color=color,ls='--',label=rf'G{group}: $s$')
pd.axvline(0,color=GREY,ls='--');pd.set(xlim=(-6,6),ylim=(0,6),xlabel=r'$E-E_{\mathrm{F}}$ (eV)',ylabel='Projected DOS')
tab(pd,'(c) Orbital projections');frame(pd);legend(fig,pd,3)
# The projected BZ segments are copied exactly; labels use the same native font size.
source=fitz.open(ROOT/'kpath_tide.pdf')[0]
for drawing in source.get_drawings():
    if drawing['color'] is None: continue
    for item in drawing['items']:
        if item[0]=='l':
            p1,p2=item[1:]
            bz.plot([p1.x,p2.x],[p1.y,p2.y],color=drawing['color'],
                    ls='--' if drawing['dashes']!='[] 0' else '-',lw=1.5)
for block in source.get_text('dict')['blocks']:
    for line in block.get('lines',[]):
        for span in line['spans']:
            color=tuple(v/255 for v in fitz.sRGB_to_rgb(span['color']))
            bz.text(*span['origin'],span['text'],fontsize=14,color=color,va='baseline')
bz.text(122.5,143,r'$\Gamma$',fontsize=14,color='#C82864')
bz.set(xlim=(0,288),ylim=(288,0),aspect='equal')
save(fig,'fig2.2.pdf')


S2.10 — H-bilayer AIMD

In [ ]:
# AIMD curves keep their full source font sizes; atomic views occupy a separate column.
rows=[]
for line in (ROOT/'6.0_AIMD/bilayer_H/OSZICAR').read_text().splitlines():
    m=re.search(r'^\s*(\d+)\s+T=\s*([\d.Ee+\-]+).*?F=\s*([\d.Ee+\-]+)',line)
    if m:rows.append([float(x) for x in m.groups()])
step,temp,energy=np.array(rows).T;assert len(step)==5500
fig=plt.figure(figsize=(10,5.4));gs=fig.add_gridspec(2,2,width_ratios=[2.1,1],left=.14,right=.99,bottom=.14,top=.95,hspace=.13,wspace=.08)
axes=[fig.add_subplot(gs[i,0]) for i in range(2)]
axes[0].plot(step/1000,energy,color=BLUE);axes[1].plot(step/1000,temp,color='#19A0A0')
axes[0].set_ylabel('Electronic free\nenergy (eV)');axes[1].set_ylabel('Temperature (K)')
axes[0].tick_params(labelbottom=False);axes[1].set_xlabel('Time (ps)');axes[1].set_ylim(0,800)
for ax in axes:ax.set_xlim(0,5.5);frame(ax)
for row,filename,title in [(0,'S2.10b1.png','Top view'),(1,'S2.10b2.png','Side view')]:
    ax=fig.add_subplot(gs[row,1]);ax.imshow(plt.imread(ROOT/'figures_collection'/filename));ax.axis('off');tab(ax,title)
save(fig,'S2.10.pdf')


Structural artwork — fig2.1, fig2.3, fig2.4, fig2.7 and S2.17
Original XCF artwork is retained; only its text is replaced. Width 8 in matches 0.8\linewidth, and 7.5 in matches 0.75\linewidth.

In [ ]:
jobs = [
    ("fig2.1", THESIS / "fig2.1/fig2.1_6k.xcf", 8.0),
    ("fig2.3", THESIS / "fig2.3/fig2.3_4k.xcf", 8.0),
    ("fig2.4", THESIS / "fig2.4/fig2.4.xcf", 8.0),
    ("fig2.7", THESIS / "fig2.7/fig2.7_from2.6.xcf", 7.5),
    ("S2.17", ROOT / "figures_collection/S2.17/S2.17_template.xcf", 7.5),
]
box = dict(boxstyle="round", facecolor="white",
           edgecolor=plt.rcParams["legend.edgecolor"],
           alpha=plt.rcParams["legend.framealpha"])
manifest = {}
with tempfile.TemporaryDirectory(prefix="thesis-structure-") as temporary:
    work = Path(temporary)
    (work / "jobs.json").write_text(json.dumps([
        {"name": name, "source": str(source)} for name, source, _ in jobs]))
    environment = os.environ.copy()
    environment["THESIS_STRUCTURE_TEMP"] = temporary
    helper = OUT / "export_structure_layers.py"
    subprocess.run(["gimp", "-n", "-i", "-d", "-f",
                    "--batch-interpreter=python-fu-eval",
                    "-b", f"exec(open({str(helper)!r}).read())", "--quit"],
                   env=environment, check=True)
    for name, source, width in jobs:
        pixels = Image.open(work / (name + ".png"))
        w, h = pixels.size
        labels = json.loads((work / (name + ".json")).read_text())
        bottom_space = .07 if name == "fig2.7" else 0
        fig = plt.figure(figsize=(width, width * h * (1 + bottom_space) / w))
        ax = fig.add_axes([0, 0, 1, 1])
        ax.imshow(pixels, interpolation="none")
        if bottom_space:
            ax.set_ylim(h * (1 + bottom_space), -.5)
        ax.set_axis_off()
        for label in labels:
            text = label["text"]
            if not text:
                continue
            x, y, align = label["x"], label["y"], "left"
            is_title = text.startswith("(") or "states along" in text
            if name in ["fig2.3", "fig2.4"]:
                column = 0 if text in ["(a)", "(c)"] else 1
                row = 0 if text in ["(a)", "(b)"] else 1
                x = (0.025 + 0.5 * column) * w
                y = 0.025 * h + row * (2400 if name == "fig2.3" else 3000)
            if name in ["fig2.7", "S2.17"]:
                if "states along" in text:
                    text = text.replace(" along T-Y", "")
                    x = 0.07 * w
                elif text in ["spin-up", "spin-down"]:
                    x = (0.22 if text == "spin-up" else 0.52) * w
                    y = (0.445 if y < h / 2 else 0.925) * h
                    if name == "fig2.7" and y > h / 2:
                        y = 1.008 * h
                    align = "center"
                elif text in ["(a)", "(b)"]:
                    x = (0.035 if text == "(a)" else 0.71) * w
            ax.text(x, y, text, fontsize=12 if is_title else 14, ha=align,
                    va="top", bbox=box if is_title else None)
        save(fig, name + ".pdf")
        manifest[name] = {
            "source": str(source), "source_sha256": hashlib.sha256(source.read_bytes()).hexdigest(),
            "canvas_width_inches": width, "tex_width_fraction": width / 10,
            "labels": labels,
        }
(OUT / "structure_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")


Verify source identities and exported PDFs

In [ ]:
for script in ['verify_spin_dos.py', 'provenance.py']:
    result = subprocess.run([sys.executable, str(OUT / script)], check=True,
                            capture_output=True, text=True,
                            env=dict(os.environ, PYTHONDONTWRITEBYTECODE='1'))
    print(result.stdout.strip())
for name in ['fig2.1.pdf','fig2.3.pdf','fig2.4.pdf','fig2.7.pdf','S2.17.pdf']:
    assert (OUT / name).read_bytes() == (THESIS / name).read_bytes()
print('All 28 PDFs match their copies in the thesis.')
